### 1. Imports and Configuration

In [1]:
import sys
import logging
from pathlib import Path

# -----------------------------------------------------------------------
# Ensure the project root is on sys.path so `tools` is importable
# regardless of where the notebook server was launched from.
# -----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger(__name__)

### 2. Import All Tools

In [2]:
from tools.mortgage         import (calculate_monthly_payment,
                                    calculate_amortization_schedule,
                                    calculate_affordability)
from tools.risk_assessment  import (calculate_credit_risk_score,
                                    assess_loan_risk)
from tools.loan             import (calculate_dti_ratio,
                                    check_loan_eligibility,
                                    compare_loans)
from tools.investment       import (calculate_compound_interest,
                                    calculate_roi,
                                    calculate_savings_goal)
from tools                  import ALL_TOOLS

log.info("loaded %d tools from tools package", len(ALL_TOOLS))

22:58:57  INFO      loaded 11 tools from tools package


### 3. Mortgage Tools

In [3]:
# -----------------------------------------------------------------------
# Monthly payment for a 400k property, 20% down, 6.5% rate, 30yr term.
# -----------------------------------------------------------------------

payment = calculate_monthly_payment(
    principal    = 400_000,
    annual_rate  = 6.5,
    term_years   = 30,
    down_payment = 80_000,
)
log.info("monthly payment result:")
for k, v in payment.items():
    log.info("  %-20s %s", k, v)

22:58:57  INFO      monthly payment result:
22:58:57  INFO        principal            400000
22:58:57  INFO        down_payment         80000
22:58:57  INFO        loan_amount          320000
22:58:57  INFO        annual_rate          6.5
22:58:57  INFO        term_years           30
22:58:57  INFO        monthly_payment      2022.62
22:58:57  INFO        total_paid           728142.36
22:58:57  INFO        total_interest       408142.36
22:58:57  INFO        loan_to_value        0.8


In [4]:
# -----------------------------------------------------------------------
# Amortization schedule -- yearly summary for the same loan.
# -----------------------------------------------------------------------

amort = calculate_amortization_schedule(
    loan_amount = payment["loan_amount"],
    annual_rate = 6.5,
    term_years  = 30,
)
log.info("amortization: %d-year schedule, monthly payment = %.2f",
         amort["term_years"], amort["monthly_payment"])
log.info("first 5 years:")
for row in amort["schedule"][:5]:
    log.info("  year %2d  principal=%10.2f  interest=%10.2f  balance=%12.2f",
             row["year"], row["principal_paid"], row["interest_paid"], row["remaining_balance"])

22:58:57  INFO      amortization: 30-year schedule, monthly payment = 2022.62
22:58:57  INFO      first 5 years:
22:58:57  INFO        year  1  principal=   3576.72  interest=  20694.69  balance=   316423.28
22:58:57  INFO        year  2  principal=   3816.26  interest=  20455.15  balance=   312607.02
22:58:57  INFO        year  3  principal=   4071.84  interest=  20199.57  balance=   308535.17
22:58:57  INFO        year  4  principal=   4344.54  interest=  19926.87  balance=   304190.63
22:58:57  INFO        year  5  principal=   4635.50  interest=  19635.91  balance=   299555.13


In [5]:
# -----------------------------------------------------------------------
# Affordability -- max home price given income and existing debts.
# -----------------------------------------------------------------------

afford = calculate_affordability(
    annual_income = 120_000,
    monthly_debts = 600,
    annual_rate   = 6.5,
    term_years    = 30,
    down_payment  = 50_000,
)
log.info("affordability result:")
for k, v in afford.items():
    log.info("  %-35s %s", k, v)

22:58:57  INFO      affordability result:
22:58:57  INFO        annual_income                       120000
22:58:57  INFO        monthly_debts                       600
22:58:57  INFO        max_monthly_mortgage_payment        3700.0
22:58:57  INFO        max_loan_amount                     585380.03
22:58:57  INFO        max_home_price                      635380.03
22:58:57  INFO        down_payment                        50000
22:58:57  INFO        annual_rate                         6.5
22:58:57  INFO        term_years                          30
22:58:57  INFO        resulting_dti                       43.0


### 4. Loan Tools

In [6]:
# -----------------------------------------------------------------------
# DTI ratio -- before and after adding a proposed mortgage payment.
# -----------------------------------------------------------------------

dti = calculate_dti_ratio(
    annual_income          = 95_000,
    monthly_debts          = 800,
    proposed_monthly_payment = payment["monthly_payment"],
)
log.info("DTI result:")
for k, v in dti.items():
    log.info("  %-40s %s", k, v)

22:58:57  INFO      DTI result:
22:58:57  INFO        monthly_income                           7916.67
22:58:57  INFO        monthly_debts                            800
22:58:57  INFO        proposed_monthly_payment                 2022.62
22:58:57  INFO        current_dti                              10.11
22:58:57  INFO        projected_dti                            35.65
22:58:57  INFO        remaining_monthly_capacity_at_43pct      581.55
22:58:57  INFO        assessment                               GOOD


In [7]:
# -----------------------------------------------------------------------
# Loan eligibility -- check Conventional / FHA / VA programs.
# -----------------------------------------------------------------------

elig = check_loan_eligibility(
    credit_score       = 720,
    annual_income      = 95_000,
    monthly_debts      = 800,
    loan_amount        = 320_000,
    property_value     = 400_000,
    is_first_time_buyer = True,
    is_veteran          = False,
)
log.info("best option: %s", elig["best_option"])
for prog, details in elig["programs"].items():
    log.info("  %-15s eligible=%-5s  issues=%s",
             prog, details["eligible"], details["issues"])

22:58:57  INFO      best option: conventional
22:58:57  INFO        conventional    eligible=True   issues=[]
22:58:57  INFO        fha             eligible=True   issues=[]
22:58:57  INFO        va              eligible=False  issues=['VA loans require veteran or active military status']


In [8]:
# -----------------------------------------------------------------------
# Loan comparison -- 3 rates x 2 terms = 6 scenarios.
# -----------------------------------------------------------------------

comp = compare_loans(
    loan_amount = 320_000,
    rates       = [5.5, 6.0, 6.5],
    terms       = [15, 30],
)
log.info("loan comparison (%d scenarios):", len(comp["scenarios"]))
for s in comp["scenarios"]:
    log.info("  %4.1f%% / %2dyr  monthly=%8.2f  total_interest=%10.2f",
             s["annual_rate"], s["term_years"], s["monthly_payment"], s["total_interest"])
log.info("cheapest monthly: %(rate)s%% / %(term)syr @ %(monthly_payment)s/mo",
         comp["cheapest_monthly_payment"])
log.info("cheapest total:   %(rate)s%% / %(term)syr @ %(total_paid)s total",
         comp["cheapest_total_cost"])

22:58:57  INFO      loan comparison (6 scenarios):
22:58:57  INFO         5.5% / 15yr  monthly= 2614.67  total_interest= 150640.07
22:58:57  INFO         5.5% / 30yr  monthly= 1816.92  total_interest= 334092.93
22:58:57  INFO         6.0% / 15yr  monthly= 2700.34  total_interest= 166061.53
22:58:57  INFO         6.0% / 30yr  monthly= 1918.56  total_interest= 370682.20
22:58:57  INFO         6.5% / 15yr  monthly= 2787.54  total_interest= 181757.84
22:58:57  INFO         6.5% / 30yr  monthly= 2022.62  total_interest= 408142.36
22:58:57  INFO      cheapest monthly: 5.5% / 30yr @ 1816.92/mo
22:58:57  INFO      cheapest total:   5.5% / 15yr @ 470640.07 total


### 5. Risk Assessment Tools

In [9]:
# -----------------------------------------------------------------------
# Credit risk score -- composite 0-100 from weighted components.
# -----------------------------------------------------------------------

risk = calculate_credit_risk_score(
    credit_score           = 720,
    dti_ratio              = dti["projected_dti"],
    ltv_ratio              = payment["loan_to_value"],
    employment_years       = 4,
    has_bankruptcy         = False,
    missed_payments_last_2y = 1,
)
log.info("credit risk: %d/100  category=%s", risk["composite_score"], risk["risk_category"])
for name, scores in risk["component_scores"].items():
    log.info("  %-25s %2d / %2d", name, scores["score"], scores["max"])
if risk["flags"]:
    log.info("flags: %s", risk["flags"])

22:58:57  INFO      credit risk: 73/100  category=MODERATE_RISK
22:58:57  INFO        credit_history            28 / 35
22:58:57  INFO        debt_to_income            15 / 25
22:58:57  INFO        loan_to_value             15 / 20
22:58:57  INFO        employment_stability       7 / 10
22:58:57  INFO        derogatory_marks           8 / 10
22:58:57  INFO      flags: ['1 missed payment(s) in last 2 years']


In [10]:
# -----------------------------------------------------------------------
# Loan risk assessment -- holistic pass/fail using mortgage outputs.
# This demonstrates tool chaining: we feed the monthly payment and
# loan amount computed earlier into the risk evaluator.
# -----------------------------------------------------------------------

loan_risk = assess_loan_risk(
    loan_amount    = payment["loan_amount"],
    property_value = 400_000,
    annual_income  = 95_000,
    monthly_debts  = 800,
    credit_score   = 720,
    monthly_payment = payment["monthly_payment"],
)
log.info("recommendation: %s  risk_level=%s",
         loan_risk["approval_recommendation"], loan_risk["overall_risk_level"])
for metric, detail in loan_risk["metrics"].items():
    log.info("  %-25s value=%-10s status=%s", metric, detail["value"], detail["status"])
log.info("requires PMI: %s", loan_risk["requires_pmi"])
if loan_risk["strengths"]:
    log.info("strengths: %s", loan_risk["strengths"])
if loan_risk["issues"]:
    log.info("issues: %s", loan_risk["issues"])

22:58:57  INFO      recommendation: APPROVE  risk_level=LOW
22:58:57  INFO        dti_with_mortgage         value=35.65      status=GOOD
22:58:57  INFO        ltv                       value=0.8        status=GOOD
22:58:57  INFO        payment_to_income         value=25.55      status=GOOD
22:58:57  INFO        credit_score              value=720        status=GOOD
22:58:57  INFO      requires PMI: False
22:58:57  INFO      strengths: ['Strong DTI of 35.7%', 'Solid equity position with 20% down', 'Housing payment well within income', 'Strong credit score']


### 6. Risk Edge Cases

In [11]:
# -----------------------------------------------------------------------
# High-risk applicant -- bankruptcy, missed payments, low credit.
# Verifies that all flags fire and the score bottoms out correctly.
# -----------------------------------------------------------------------

high_risk = calculate_credit_risk_score(
    credit_score           = 500,
    dti_ratio              = 55.0,
    ltv_ratio              = 0.98,
    employment_years       = 0.5,
    has_bankruptcy         = True,
    missed_payments_last_2y = 3,
)
log.info("high-risk applicant: %d/100  category=%s",
         high_risk["composite_score"], high_risk["risk_category"])
log.info("flags (%d): %s", len(high_risk["flags"]), high_risk["flags"])

22:58:57  INFO      high-risk applicant: 9/100  category=HIGH_RISK
22:58:57  INFO      flags (6): ['Very low credit score', 'DTI exceeds conventional loan limits', 'Very high LTV - likely requires PMI', 'Less than 1 year at current employer', 'Bankruptcy on record', '3 missed payment(s) in last 2 years']


In [12]:
# -----------------------------------------------------------------------
# Perfect applicant -- no flags, maximum score.
# -----------------------------------------------------------------------

perfect = calculate_credit_risk_score(
    credit_score           = 800,
    dti_ratio              = 15.0,
    ltv_ratio              = 0.50,
    employment_years       = 10,
    has_bankruptcy         = False,
    missed_payments_last_2y = 0,
)
log.info("perfect applicant: %d/100  category=%s",
         perfect["composite_score"], perfect["risk_category"])
assert perfect["composite_score"] == 100, "perfect applicant should score 100"
assert perfect["flags"] == [], "perfect applicant should have no flags"
log.info("assertions passed")

22:58:57  INFO      perfect applicant: 100/100  category=LOW_RISK
22:58:57  INFO      assertions passed


### 7. Investment Tools

In [13]:
# -----------------------------------------------------------------------
# Compound interest -- 10k initial + 500/mo for 20 years at 7%.
# -----------------------------------------------------------------------

growth = calculate_compound_interest(
    principal            = 10_000,
    annual_rate          = 7.0,
    years                = 20,
    monthly_contribution = 500,
)
log.info("compound interest over %d years:", growth["years"])
log.info("  future value:       %12.2f", growth["future_value"])
log.info("  total contributed:  %12.2f", growth["total_contributions"])
log.info("  interest earned:    %12.2f", growth["total_interest_earned"])
log.info("  year-by-year (first 5):")
for row in growth["yearly_summary"][:5]:
    log.info("    year %2d  balance=%12.2f  interest=%10.2f",
             row["year"], row["balance"], row["interest_earned"])

22:58:57  INFO      compound interest over 20 years:
22:58:57  INFO        future value:          300850.72
22:58:57  INFO        total contributed:     130000.00
22:58:57  INFO        interest earned:       170850.72
22:58:57  INFO        year-by-year (first 5):
22:58:57  INFO          year  1  balance=    16919.19  interest=    919.19
22:58:57  INFO          year  2  balance=    24338.58  interest=   2338.58
22:58:57  INFO          year  3  balance=    32294.31  interest=   4294.31
22:58:57  INFO          year  4  balance=    40825.16  interest=   6825.16
22:58:57  INFO          year  5  balance=    49972.70  interest=   9972.70


In [14]:
# -----------------------------------------------------------------------
# ROI -- bought at 250k, now worth 340k after 5 years, 8k/yr rent.
# -----------------------------------------------------------------------

roi = calculate_roi(
    initial_investment = 250_000,
    final_value        = 340_000,
    years              = 5,
    annual_cash_flows  = 8_000,
)
log.info("ROI result:")
for k, v in roi.items():
    log.info("  %-25s %s", k, v)

22:58:57  INFO      ROI result:
22:58:57  INFO        initial_investment        250000
22:58:57  INFO        final_value               340000
22:58:57  INFO        total_cash_flows          40000
22:58:57  INFO        total_gain                130000
22:58:57  INFO        simple_roi_pct            52.0
22:58:57  INFO        annualized_return_pct     8.73
22:58:57  INFO        years                     5


In [15]:
# -----------------------------------------------------------------------
# Savings goal -- how long to reach 100k starting from 15k at 750/mo.
# -----------------------------------------------------------------------

goal = calculate_savings_goal(
    target_amount        = 100_000,
    current_savings      = 15_000,
    annual_rate          = 5.0,
    monthly_contribution = 750,
)
log.info("savings goal result:")
for k, v in goal.items():
    log.info("  %-30s %s", k, v)

22:58:57  INFO      savings goal result:
22:58:57  INFO        target_amount                  100000
22:58:57  INFO        current_savings                15000
22:58:57  INFO        monthly_contribution           750
22:58:57  INFO        annual_rate                    5.0
22:58:57  INFO        months_to_goal                 88
22:58:57  INFO        years_to_goal                  7.3
22:58:57  INFO        total_contributed              81000
22:58:57  INFO        interest_earned                20154.53


In [16]:
# -----------------------------------------------------------------------
# Savings goal (reverse) -- how much per month to hit 100k in 10 years.
# -----------------------------------------------------------------------

goal_rev = calculate_savings_goal(
    target_amount   = 100_000,
    current_savings = 15_000,
    annual_rate     = 5.0,
)
log.info("required monthly contribution: %.2f", goal_rev["required_monthly_contribution"])
log.info("over %d years, total contributed: %.2f, interest earned: %.2f",
         goal_rev["assumed_timeline_years"],
         goal_rev["total_contributed_over_period"],
         goal_rev["projected_interest_earned"])

22:58:57  INFO      required monthly contribution: 484.89
22:58:57  INFO      over 10 years, total contributed: 73186.83, interest earned: 26813.17


### 8. Tool Chaining Scenario

In [17]:
# -----------------------------------------------------------------------
# End-to-end scenario: a prospective buyer wants to know if they can
# afford a 500k home. Chaining affordability -> mortgage -> DTI ->
# risk assessment -> eligibility, passing outputs forward each step.
# This is the kind of pipeline an agent would execute.
# -----------------------------------------------------------------------

INCOME       = 130_000
DEBTS        = 500
DOWN_PAYMENT = 100_000
RATE         = 6.25
CREDIT       = 740
EMP_YEARS    = 6
PROPERTY     = 500_000

log.info("--- step 1: affordability check ---")
step1 = calculate_affordability(
    annual_income = INCOME,
    monthly_debts = DEBTS,
    annual_rate   = RATE,
    down_payment  = DOWN_PAYMENT,
)
log.info("max home price: %.2f  (target: %d)", step1["max_home_price"], PROPERTY)
can_afford = step1["max_home_price"] >= PROPERTY
log.info("within budget: %s", can_afford)

log.info("--- step 2: mortgage calculation ---")
step2 = calculate_monthly_payment(
    principal    = PROPERTY,
    annual_rate  = RATE,
    term_years   = 30,
    down_payment = DOWN_PAYMENT,
)
log.info("loan amount: %.2f  monthly payment: %.2f", step2["loan_amount"], step2["monthly_payment"])

log.info("--- step 3: DTI ratio ---")
step3 = calculate_dti_ratio(
    annual_income           = INCOME,
    monthly_debts           = DEBTS,
    proposed_monthly_payment = step2["monthly_payment"],
)
log.info("projected DTI: %.2f%%  assessment: %s", step3["projected_dti"], step3["assessment"])

log.info("--- step 4: credit risk score ---")
step4 = calculate_credit_risk_score(
    credit_score     = CREDIT,
    dti_ratio        = step3["projected_dti"],
    ltv_ratio        = step2["loan_to_value"],
    employment_years = EMP_YEARS,
)
log.info("risk score: %d/100  category: %s", step4["composite_score"], step4["risk_category"])

log.info("--- step 5: loan risk assessment ---")
step5 = assess_loan_risk(
    loan_amount     = step2["loan_amount"],
    property_value  = PROPERTY,
    annual_income   = INCOME,
    monthly_debts   = DEBTS,
    credit_score    = CREDIT,
    monthly_payment = step2["monthly_payment"],
)
log.info("recommendation: %s  risk: %s",
         step5["approval_recommendation"], step5["overall_risk_level"])

log.info("--- step 6: program eligibility ---")
step6 = check_loan_eligibility(
    credit_score        = CREDIT,
    annual_income       = INCOME,
    monthly_debts       = DEBTS,
    loan_amount         = step2["loan_amount"],
    property_value      = PROPERTY,
    is_first_time_buyer = True,
)
log.info("best program: %s", step6["best_option"])
for prog, det in step6["programs"].items():
    log.info("  %-15s eligible=%s", prog, det["eligible"])

22:58:57  INFO      --- step 1: affordability check ---
22:58:57  INFO      max home price: 775364.17  (target: 500000)
22:58:57  INFO      within budget: True
22:58:57  INFO      --- step 2: mortgage calculation ---
22:58:57  INFO      loan amount: 400000.00  monthly payment: 2462.87
22:58:57  INFO      --- step 3: DTI ratio ---
22:58:57  INFO      projected DTI: 27.35%  assessment: GOOD
22:58:57  INFO      --- step 4: credit risk score ---
22:58:57  INFO      risk score: 83/100  category: LOW_RISK
22:58:57  INFO      --- step 5: loan risk assessment ---
22:58:57  INFO      recommendation: APPROVE  risk: LOW
22:58:57  INFO      --- step 6: program eligibility ---
22:58:57  INFO      best program: conventional
22:58:57  INFO        conventional    eligible=True
22:58:57  INFO        fha             eligible=True
22:58:57  INFO        va              eligible=False


### 9. Error Handling

In [18]:
# -----------------------------------------------------------------------
# Verify that tools return error dicts for invalid inputs rather
# than raising exceptions. Agents need graceful failures to inform user.
# -----------------------------------------------------------------------

err1 = calculate_monthly_payment(400_000, 6.5, 30, down_payment=500_000)
assert "error" in err1, "expected error for down payment > principal"
log.info("down payment > principal: %s", err1["error"])

err2 = calculate_monthly_payment(400_000, -1.0, 30)
assert "error" in err2, "expected error for negative rate"
log.info("negative rate:           %s", err2["error"])

err3 = calculate_dti_ratio(-50_000, 800)
assert "error" in err3, "expected error for negative income"
log.info("negative income:         %s", err3["error"])

err4 = calculate_affordability(60_000, 5_000, 6.5)
assert "error" in err4, "expected error when debts exceed DTI threshold"
log.info("debts exceed DTI:        %s", err4["error"])

log.info("all error-handling assertions passed")

22:58:57  INFO      down payment > principal: Down payment exceeds or equals property price.
22:58:57  INFO      negative rate:           Interest rate cannot be negative.
22:58:57  INFO      negative income:         Annual income must be positive.
22:58:57  INFO      debts exceed DTI:        Existing debts already exceed the maximum DTI threshold.
22:58:57  INFO      all error-handling assertions passed


### 10. Tool Metadata Inspection

In [19]:
# -----------------------------------------------------------------------
# ADK uses function names, docstrings, and type hints to build the
# tool schema presented to the model. Verify that every tool has
# the metadata the framework needs.
# -----------------------------------------------------------------------

import inspect

for tool in ALL_TOOLS:
    sig    = inspect.signature(tool)
    doc    = tool.__doc__ or ""
    params = list(sig.parameters.keys())

    assert doc.strip(), f"{tool.__name__} is missing a docstring"
    assert sig.return_annotation != inspect.Parameter.empty or "->" in str(sig), \
        f"{tool.__name__} is missing a return type hint"

    log.info("%-40s params=%-60s has_docstring=%s",
             tool.__name__, str(params), bool(doc.strip()))

log.info("all %d tools have docstrings and type hints", len(ALL_TOOLS))

22:58:57  INFO      calculate_monthly_payment                params=['principal', 'annual_rate', 'term_years', 'down_payment']   has_docstring=True
22:58:57  INFO      calculate_amortization_schedule          params=['loan_amount', 'annual_rate', 'term_years', 'num_periods']  has_docstring=True
22:58:57  INFO      calculate_affordability                  params=['annual_income', 'monthly_debts', 'annual_rate', 'term_years', 'max_dti', 'down_payment'] has_docstring=True
22:58:57  INFO      calculate_credit_risk_score              params=['credit_score', 'dti_ratio', 'ltv_ratio', 'employment_years', 'has_bankruptcy', 'missed_payments_last_2y'] has_docstring=True
22:58:57  INFO      assess_loan_risk                         params=['loan_amount', 'property_value', 'annual_income', 'monthly_debts', 'credit_score', 'monthly_payment'] has_docstring=True
22:58:57  INFO      calculate_dti_ratio                      params=['annual_income', 'monthly_debts', 'proposed_monthly_payment'] has_docstr